# Procesar Data y Almacenar en GCS

In [ ]:
# from google.colab import auth
# auth.authenticate_user()

Extraemos data ficticia de una API

In [1]:
import requests
size = '2'
URL = 'https://randomuser.me/api/?results=' + size
response = requests.get(URL)
# response
data = response.json()
data['results']

[{'gender': 'female',
  'name': {'title': 'Miss', 'first': 'Amelia', 'last': 'Bates'},
  'location': {'street': {'number': 3406, 'name': 'Strand Road'},
   'city': 'Tullow',
   'state': 'South Dublin',
   'country': 'Ireland',
   'postcode': 24403,
   'coordinates': {'latitude': '-1.7807', 'longitude': '45.1924'},
   'timezone': {'offset': '-8:00',
    'description': 'Pacific Time (US & Canada)'}},
  'email': 'amelia.bates@example.com',
  'login': {'uuid': '10525cfd-29e4-4a7d-bec2-b8560fa959c7',
   'username': 'blackpanda602',
   'password': 'scorpion',
   'salt': 'L5pPEZ75',
   'md5': '24e3de8a99f775459b89a65b57c1d985',
   'sha1': '6a434c3f04c92959ad853a10c1245095b6f9ff72',
   'sha256': '4e8a6864645d81f62819e139491672a3075137cedff37bf398aa8d1efb0d75b1'},
  'dob': {'date': '1963-07-05T19:55:39.602Z', 'age': 62},
  'registered': {'date': '2010-07-28T00:06:46.398Z', 'age': 15},
  'phone': '071-653-6745',
  'cell': '081-787-5686',
  'id': {'name': 'PPS', 'value': '3214794T'},
  'picture':

Pasamos a DataFrame

In [2]:
import pandas as pd

df = pd.DataFrame(data['results'])
df

,gender,name,location,email,login,dob,registered,phone,cell,id,picture,nat
0,female,"{'title': 'Miss', 'first': 'Amelia', 'last': '...","{'street': {'number': 3406, 'name': 'Strand Ro...",amelia.bates@example.com,{'uuid': '10525cfd-29e4-4a7d-bec2-b8560fa959c7...,"{'date': '1963-07-05T19:55:39.602Z', 'age': 62}","{'date': '2010-07-28T00:06:46.398Z', 'age': 15}",071-653-6745,081-787-5686,"{'name': 'PPS', 'value': '3214794T'}",{'large': 'https://randomuser.me/api/portraits...,IE
1,male,"{'title': 'Mr', 'first': 'Gudo', 'last': 'Pepp...","{'street': {'number': 6808, 'name': 'Goejanver...",gudo.peppelman@example.com,{'uuid': 'dc9ad953-26be-47b7-98da-9656aa5ff71e...,"{'date': '1964-05-24T06:43:05.558Z', 'age': 61}","{'date': '2018-06-10T11:28:53.004Z', 'age': 7}",(016) 2957075,(06) 17381476,"{'name': 'BSN', 'value': '20891682'}",{'large': 'https://randomuser.me/api/portraits...,NL


In [3]:
rows = []
for index, row in df.iterrows():
    values = {
        'id':row['id']['value'],
        'first_name':row['name']['first'],
        'last_name':row['name']['last'],
        'username':row['login']['username'],
        'email':row['email'],
        'phone_number':row['cell'],
        'date_of_birth':row['dob']['date'],
        'employment_title': row['name']['title'],
        'address_street': row['location']['street']['name'] + ' ' + str(row['location']['street']['number']),
        'address_city': row['location']['city'],
        'address_state': row['location']['state'],
        'address_country': row['location']['country']
    }
    rows.append(values)

df_processed = pd.DataFrame(rows)
df_processed

,id,first_name,last_name,username,email,phone_number,date_of_birth,employment_title,address_street,address_city,address_state,address_country
0,3214794T,Amelia,Bates,blackpanda602,amelia.bates@example.com,081-787-5686,1963-07-05T19:55:39.602Z,Miss,Strand Road 3406,Tullow,South Dublin,Ireland
1,20891682,Gudo,Peppelman,redsnake414,gudo.peppelman@example.com,(06) 17381476,1964-05-24T06:43:05.558Z,Mr,Goejanverwelledijk 6808,Idaard,Groningen,Netherlands


In [4]:
df.columns

Index(['gender', 'name', 'location', 'email', 'login', 'dob', 'registered',
       'phone', 'cell', 'id', 'picture', 'nat'],
      dtype='str')

In [5]:
df_processed.columns

Index(['id', 'first_name', 'last_name', 'username', 'email', 'phone_number',
       'date_of_birth', 'employment_title', 'address_street', 'address_city',
       'address_state', 'address_country'],
      dtype='str')

## Función para ingesta

In [23]:
from datetime import datetime
import uuid
import random
from google.cloud import storage
import json
import requests

project_id = 'lustrous-setup-457000-h5'
bucket_name = 'sd-gcolab-bucket-01'

def function_ingest_data():
    date_str = datetime.now().strftime('%Y%m%d')
    uuid_4dig = str(uuid.uuid4().hex)[:4]

    destination_blob_name = f'ingested/json/web_users/{date_str}/web_users_{uuid_4dig}.json'

    size = str(random.randint(3,10))
    URL = 'https://randomuser.me/api/?results=' + size
    response = requests.get(URL)
    data = response.json()
    print(data['results'])
    storage_client = storage.Client(project=project_id)
    bucket = storage_client.get_bucket(bucket_name)
    blob = bucket.blob(destination_blob_name)
    data_str = json.dumps(data['results'])
    blob.upload_from_string(data_str)
    print(destination_blob_name)

    message = f'Object created at {bucket_name} with name "{destination_blob_name}".'
    print(message)
    return json.dumps({"message": message})

In [24]:
function_ingest_data()

[{'gender': 'female', 'name': {'title': 'Miss', 'first': 'آیناز', 'last': 'موسوی'}, 'location': {'street': {'number': 8950, 'name': 'شهید بهشتی'}, 'city': 'نیشابور', 'state': 'گلستان', 'country': 'Iran', 'postcode': 19322, 'coordinates': {'latitude': '18.3830', 'longitude': '177.1189'}, 'timezone': {'offset': '-9:00', 'description': 'Alaska'}}, 'email': 'aynz.mwswy@example.com', 'login': {'uuid': '2a01e6ce-1d24-4264-ae85-b513ea6c3fe7', 'username': 'greenpeacock387', 'password': 'leng', 'salt': '1lsjArP5', 'md5': 'ec23e7e55268aa4555014d6599e1215e', 'sha1': '0b782a3eab9ca73119f07f5282e2451ddbb5f72c', 'sha256': '493817b4e7753ff0a82509bba0937c0e63f6032790eaff4a1765d3f442c53c8e'}, 'dob': {'date': '1948-11-02T14:53:30.070Z', 'age': 76}, 'registered': {'date': '2006-12-19T06:08:09.789Z', 'age': 18}, 'phone': '064-82994884', 'cell': '0917-727-0077', 'id': {'name': '', 'value': None}, 'picture': {'large': 'https://randomuser.me/api/portraits/women/89.jpg', 'medium': 'https://randomuser.me/api/p

'{"message": "Object created at sd-gcolab-bucket-01 with name \\"ingested/json/web_users/20250526/web_users_e6f7.json\\"."}'

## Función para procesamiento

In [25]:
import pandas as pd
import json
from google.cloud import storage
from datetime import datetime

project_id = 'lustrous-setup-457000-h5'
bucket_name = 'sd-gcolab-bucket-01'

def function_transform_data():

    client = storage.Client()
    bucket = client.get_bucket(bucket_name)
    date_str = datetime.now().strftime('%Y%m%d')
    prefix_source = 'ingested/json/web_users/' + date_str
    prefix_target = 'transformed/csv/web_users/' + date_str
    blobs = bucket.list_blobs(prefix=prefix_source)
    blob_names = [blob.name.split("/")[-1] for blob in blobs]
    print(blob_names)

    rows = []

    for blob in blob_names:
        source_path = 'gs://' + bucket_name + '/' + prefix_source + '/' + blob
        print(source_path)
        target_path = 'gs://' + bucket_name + '/' + prefix_target + '/' + 'web_users.csv'
        print(target_path)
        df = pd.read_json(source_path)
        print(df)

        for index, row in df.iterrows():
            values = {
                'id':row['id']['value'],
                'first_name':row['name']['first'],
                'last_name':row['name']['last'],
                'username':row['login']['username'],
                'email':row['email'],
                'phone_number':row['cell'],
                'date_of_birth':row['dob']['date'],
                'employment_title': row['name']['title'],
                'address_street': row['location']['street']['name'] + ' ' + str(row['location']['street']['number']),
                'address_city': row['location']['city'],
                'address_state': row['location']['state'],
                'address_country': row['location']['country'],
            }
            rows.append(values)

    df_silver = pd.DataFrame(rows)
    print(df_silver)

    message = f'Objects transformed: {source_path} and loaded as: {target_path}'
    print(message)
    df_silver.to_csv(target_path, index=False)

In [26]:
function_transform_data()

['web_users_e6f7.json']
gs://sd-gcolab-bucket-01/ingested/json/web_users/20250526/web_users_e6f7.json
gs://sd-gcolab-bucket-01/transformed/csv/web_users/20250526/web_users.csv
   gender                                               name  \
0  female  {'title': 'Miss', 'first': 'آیناز', 'last': 'م...   
1    male  {'title': 'Mr', 'first': 'Howard', 'last': 'Mi...   
2    male  {'title': 'Mr', 'first': 'Joris', 'last': 'Den...   

                                            location  \
0  {'street': {'number': 8950, 'name': 'شهید بهشت...   
1  {'street': {'number': 3957, 'name': 'York Road...   
2  {'street': {'number': 4928, 'name': 'Rue de Ge...   

                         email  \
0       aynz.mwswy@example.com   
1  howard.mitchell@example.com   
2      joris.denis@example.com   

                                               login  \
0  {'uuid': '2a01e6ce-1d24-4264-ae85-b513ea6c3fe7...   
1  {'uuid': '72bc3b01-bc71-469c-afea-1896ae78095a...   
2  {'uuid': '79074c3b-3951-423c-a3d6-

In [ ]:
function_ingest_data()
function_transform_data()